In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ruptures scipy matplotlib -q

import json
import numpy as np
import os
import ruptures as rpt
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DRIVE_BASE = "/content/drive/MyDrive/medrag"
OUTPUT_DIR = f"{DRIVE_BASE}/eas_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_trial(trial_num, filename):
    path = f"{DRIVE_BASE}/trial{trial_num}_outputs/{filename}"
    with open(path, "r") as f:
        return json.load(f)

t1 = load_trial(1, "trial1_full_results.json")
t2 = load_trial(2, "trial2_full_results.json")
t3 = load_trial(3, "trial3_full_results.json")
t4 = load_trial(4, "trial4_full_results.json")

n_layers = t1["n_layers"]

print(f"Trial 1: {t1['n_samples']} samples")
print(f"Trial 2: {t2['n_samples']} samples")
print(f"Trial 3: {t3['n_samples']} samples")
print(f"Trial 4: {t4['n_samples']} samples")
print(f"n_layers: {n_layers}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
t1_by_pubid = {r["pubid"]: r for r in t1["samples"]}
t2_by_pubid = {r["pubid"]: r for r in t2["samples"]}
t3_by_pubid = {r["pubid"]: r for r in t3["samples"]}
t4_by_pubid = {r["pubid"]: r for r in t4["samples"]}

common_pubids = (
    set(t1_by_pubid.keys())
    & set(t2_by_pubid.keys())
    & set(t3_by_pubid.keys())
    & set(t4_by_pubid.keys())
)

print(f"Common pubids across all trials: {len(common_pubids)}")
print(f"T1 exclusive: {len(set(t1_by_pubid) - common_pubids)}")
print(f"T2 exclusive: {len(set(t2_by_pubid) - common_pubids)}")
print(f"T3 exclusive: {len(set(t3_by_pubid) - common_pubids)}")
print(f"T4 exclusive: {len(set(t4_by_pubid) - common_pubids)}")

aligned = []
for r in sorted(t1["samples"], key=lambda x: x["idx"]):
    pid = r["pubid"]
    if pid in common_pubids:
        aligned.append({
            "pubid":  pid,
            "label":  r["label"],
            "idx":    r["idx"],
            "t1":     t1_by_pubid[pid],
            "t2":     t2_by_pubid[pid],
            "t3":     t3_by_pubid[pid],
            "t4":     t4_by_pubid[pid],
        })

N = len(aligned)
print(f"\nAligned samples: {N}")
print("Label breakdown:")
for label in ["yes", "no", "maybe"]:
    count = sum(1 for r in aligned if r["label"] == label)
    print(f"  {label}: {count}")

low_content_count = sum(1 for r in aligned if r["t2"].get("low_content", False))
print(f"\nLow-content samples (T2 flag): {low_content_count}/{N}")

In [ ]:
EPSILON = 1e-6

attn_arr  = np.array([r["t1"]["attention_trajectory"]  for r in aligned])
prob_arr  = np.array([r["t2"]["prob_mass_trajectory"]  for r in aligned])
t3_arr    = np.array([r["t3"]["kl_trajectory"]          for r in aligned])
t4_arr    = np.array([r["t4"]["kl_trajectory"]          for r in aligned])

ratio_arr = t4_arr / np.maximum(t3_arr, EPSILON)

print(f"Signal arrays shape: {attn_arr.shape}  (N={N}, L={n_layers})\n")
print(f"{'Signal':<22} {'mean':>8} {'std':>8} {'min':>8} {'max':>8}")
print("-" * 58)
for name, arr in [
    ("T1 attention",    attn_arr),
    ("T2 prob_mass",    prob_arr),
    ("T3 KL_uniform",   t3_arr),
    ("T4 KL_rag_sup",   t4_arr),
    ("T4/T3 ratio",     ratio_arr),
]:
    print(f"{name:<22} {arr.mean():>8.4f} {arr.std():>8.4f} "
          f"{arr.min():>8.4f} {arr.max():>8.4f}")

In [ ]:
W1 = 0.15
W2 = 0.425
W3 = 0.425

def global_zscore(arr):
    return (arr - arr.mean()) / (arr.std() + 1e-8)

z_attn  = global_zscore(attn_arr)
z_prob  = global_zscore(prob_arr)
z_ratio = global_zscore(ratio_arr)

eas_arr = W1 * z_attn + W2 * z_prob + W3 * z_ratio  # (N, L)

print(f"EAS shape: {eas_arr.shape}")
print(f"EAS global: mean={eas_arr.mean():.4f}  std={eas_arr.std():.4f}\n")
print("EAS by label:")
for label in ["yes", "no", "maybe"]:
    mask = np.array([r["label"] == label for r in aligned])
    if mask.sum() == 0:
        continue
    vals = eas_arr[mask]
    print(f"  {label}: mean={vals.mean():.4f}  std={vals.std():.4f}  n={mask.sum()}")

print("\nPer-layer mean EAS (first and last 5 layers):")
layer_means = eas_arr.mean(axis=0)
for i in list(range(5)) + list(range(n_layers - 5, n_layers)):
    print(f"  Layer {i:2d}: {layer_means[i]:.4f}")

In [ ]:
PELT_PENALTY          = 3
MIN_SEGMENT_SIZE      = 3
COLLAPSE_SD_THRESHOLD = 1.0

collapse_results = []

for i, row in enumerate(aligned):
    trajectory = eas_arr[i]
    signal_2d  = trajectory.reshape(-1, 1)

    algo = rpt.Pelt(model="l2", min_size=MIN_SEGMENT_SIZE, jump=1)
    algo.fit(signal_2d)
    breakpoints  = algo.predict(pen=PELT_PENALTY)
    changepoints = [bp for bp in breakpoints if bp < n_layers]

    collapse_detected = False
    collapse_layer    = None
    pre_mean = post_mean = drop_magnitude = None

    if changepoints:
        cp       = changepoints[-1]
        pre_seg  = trajectory[:cp]
        post_seg = trajectory[cp:]

        if len(pre_seg) > 0 and len(post_seg) > 0:
            pre_mean       = float(pre_seg.mean())
            pre_std        = float(pre_seg.std())
            post_mean      = float(post_seg.mean())
            drop_magnitude = pre_mean - post_mean
            collapse_detected = post_mean < (pre_mean - COLLAPSE_SD_THRESHOLD * pre_std)
            if collapse_detected:
                collapse_layer = cp

    collapse_results.append({
        "pubid":             row["pubid"],
        "label":             row["label"],
        "eas_trajectory":    trajectory.tolist(),
        "changepoints":      changepoints,
        "n_changepoints":    len(changepoints),
        "collapse_detected": collapse_detected,
        "collapse_layer":    collapse_layer,
        "pre_mean":          pre_mean,
        "post_mean":         post_mean,
        "drop_magnitude":    drop_magnitude,
        "mean_eas":          float(trajectory.mean()),
        "max_eas_layer":     int(np.argmax(trajectory)),
        "final_layer_eas":   float(trajectory[-1]),
        "low_content_t2":    row["t2"].get("low_content", False)
    })

total_collapse = sum(1 for r in collapse_results if r["collapse_detected"])
print(f"PELT complete.")
print(f"  model=l2  penalty={PELT_PENALTY}  "
      f"min_size={MIN_SEGMENT_SIZE}  SD_threshold={COLLAPSE_SD_THRESHOLD}\n")
print(f"Collapse detected: {total_collapse}/{N} ({total_collapse/N:.1%})\n")
print("By label:")
for label in ["yes", "no"]:
    lr = [r for r in collapse_results if r["label"] == label]
    nc = sum(1 for r in lr if r["collapse_detected"])
    cl = [r["collapse_layer"] for r in lr if r["collapse_layer"] is not None]
    mean_cl = f"{np.mean(cl):.1f}" if cl else "n/a"
    print(f"  {label}: collapse {nc}/{len(lr)} ({nc/len(lr):.1%}) | "
          f"mean collapse layer: {mean_cl}")

In [ ]:
print("=" * 65)
print("STATISTICAL TESTING: yes vs no label comparisons")
print("=" * 65)

yes_mask = np.array([r["label"] == "yes" for r in aligned])
no_mask  = np.array([r["label"] == "no"  for r in aligned])

def get_label_field(field, label):
    return [r[field] for r in collapse_results
            if r["label"] == label and r[field] is not None]

for field, description in [
    ("mean_eas",        "Mean EAS (composite, all layers)"),
    ("final_layer_eas", "Final layer EAS"),
    ("max_eas_layer",   "Peak EAS layer index"),
]:
    yes_vals = get_label_field(field, "yes")
    no_vals  = get_label_field(field, "no")
    t_stat, p_t = ttest_ind(yes_vals, no_vals, equal_var=False)
    u_stat, p_u = mannwhitneyu(yes_vals, no_vals, alternative="two-sided")
    print(f"\n{description}")
    print(f"  yes: {np.mean(yes_vals):.4f}  (n={len(yes_vals)})")
    print(f"  no:  {np.mean(no_vals):.4f}   (n={len(no_vals)})")
    print(f"  Welch t-test: t={t_stat:.3f}, p={p_t:.4f}"
          f"{'  *' if p_t < 0.05 else ''}")
    print(f"  Mann-Whitney: U={u_stat:.0f},  p={p_u:.4f}"
          f"{'  *' if p_u < 0.05 else ''}")

yes_cr = [r for r in collapse_results if r["label"] == "yes"]
no_cr  = [r for r in collapse_results if r["label"] == "no"]
yes_nc = sum(1 for r in yes_cr if r["collapse_detected"])
no_nc  = sum(1 for r in no_cr  if r["collapse_detected"])
contingency = np.array([
    [yes_nc,             len(yes_cr) - yes_nc],
    [no_nc,              len(no_cr)  - no_nc]
])
chi2, p_chi2, dof, _ = stats.chi2_contingency(contingency)
print(f"\nCollapse rate chi-square (yes vs no):")
print(f"  yes: {yes_nc}/{len(yes_cr)} ({yes_nc/len(yes_cr):.1%})")
print(f"  no:  {no_nc}/{len(no_cr)} ({no_nc/len(no_cr):.1%})")
print(f"  chi2={chi2:.3f}, dof={dof}, p={p_chi2:.4f}"
      f"{'  *' if p_chi2 < 0.05 else ''}")

print(f"\nPer-layer t-tests (EAS at each layer, yes vs no):")
sig_layers = []
for l in range(n_layers):
    t, p = ttest_ind(eas_arr[yes_mask, l], eas_arr[no_mask, l], equal_var=False)
    if p < 0.05:
        sig_layers.append((l, float(t), float(p)))

if sig_layers:
    print(f"  Significant layers (p<0.05): {[s[0] for s in sig_layers]}")
    for l, t, p in sig_layers:
        direction = "yes>no" if t > 0 else "no>yes"
        print(f"    Layer {l:2d}: t={t:+.3f}, p={p:.4f}  [{direction}]")
else:
    print("  No individually significant layers at p<0.05.")

In [ ]:
layers   = np.arange(n_layers)
yes_eas  = eas_arr[yes_mask]
no_eas   = eas_arr[no_mask]
yes_mean = yes_eas.mean(axis=0)
yes_se   = yes_eas.std(axis=0) / np.sqrt(yes_mask.sum())
no_mean  = no_eas.mean(axis=0)
no_se    = no_eas.std(axis=0)  / np.sqrt(no_mask.sum())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(layers, yes_mean, color="steelblue", label="yes", linewidth=2)
ax.fill_between(layers, yes_mean - yes_se, yes_mean + yes_se,
                color="steelblue", alpha=0.2)
ax.plot(layers, no_mean, color="tomato", label="no", linewidth=2)
ax.fill_between(layers, no_mean - no_se, no_mean + no_se,
                color="tomato", alpha=0.2)
ax.axvline(x=21, color="gray", linestyle="--", alpha=0.6, label="L21 (prob mass peak)")
ax.axvline(x=23, color="gray", linestyle=":",  alpha=0.6, label="L23 (KL peak)")
ax.axvline(x=26, color="gray", linestyle="-.", alpha=0.6, label="L26 (parametric conf.)")
ax.set_xlabel("Layer")
ax.set_ylabel("EAS (z-scored composite)")
ax.set_title("Mean EAS Trajectory by Label\n(shading = ±1 SE)")
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

ax = axes[1]
yes_indices = np.where(yes_mask)[0]
no_indices  = np.where(no_mask)[0]
yes_sorted  = yes_indices[np.argsort(-eas_arr[yes_indices].mean(axis=1))]
no_sorted   = no_indices[np.argsort(-eas_arr[no_indices].mean(axis=1))]
order       = np.concatenate([yes_sorted, no_sorted])
heatmap_data = eas_arr[order]
vmax = np.percentile(np.abs(heatmap_data), 95)
im   = ax.imshow(heatmap_data, aspect="auto", cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, interpolation="nearest")
ax.axhline(y=len(yes_sorted) - 0.5, color="white", linewidth=1.5)
ax.set_xlabel("Layer")
ax.set_ylabel("Sample (sorted by label, then mean EAS ↓)")
ax.set_title("EAS Heatmap\n(red=high adherence, blue=low)")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[2]
yes_cl = [r["collapse_layer"] for r in collapse_results
          if r["label"] == "yes" and r["collapse_layer"] is not None]
no_cl  = [r["collapse_layer"] for r in collapse_results
          if r["label"] == "no"  and r["collapse_layer"] is not None]
bins   = np.arange(0, n_layers + 1) - 0.5
ax.hist(yes_cl, bins=bins, alpha=0.65, color="steelblue",
        label=f"yes (n={len(yes_cl)})", density=True)
ax.hist(no_cl,  bins=bins, alpha=0.65, color="tomato",
        label=f"no  (n={len(no_cl)})",  density=True)
ax.set_xlabel("Collapse Layer")
ax.set_ylabel("Density")
ax.set_title("Collapse Layer Distribution by Label")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, "eas_figures.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved to {fig_path}")

In [ ]:
eas_output = {
    "n_samples": N,
    "n_layers":  n_layers,
    "weights": {"w1_attention": W1, "w2_prob_mass": W2, "w3_t4_t3_ratio": W3},
    "normalization": "global_zscore_per_signal",
    "pelt_params": {
        "model":                 "l2",
        "penalty":               PELT_PENALTY,
        "min_size":              MIN_SEGMENT_SIZE,
        "collapse_sd_threshold": COLLAPSE_SD_THRESHOLD
    },
    "summary": {
        "total_collapse_count": total_collapse,
        "total_collapse_rate":  total_collapse / N,
        "by_label": {}
    },
    "per_layer_significance": [
        {"layer": l, "t": t, "p": p} for l, t, p in sig_layers
    ],
    "samples": collapse_results
}

for label in ["yes", "no"]:
    lr = [r for r in collapse_results if r["label"] == label]
    nc = sum(1 for r in lr if r["collapse_detected"])
    cl = [r["collapse_layer"] for r in lr if r["collapse_layer"] is not None]
    eas_output["summary"]["by_label"][label] = {
        "n":                   len(lr),
        "collapse_count":      nc,
        "collapse_rate":       nc / len(lr),
        "mean_eas":            float(np.mean([r["mean_eas"]        for r in lr])),
        "mean_final_eas":      float(np.mean([r["final_layer_eas"] for r in lr])),
        "mean_collapse_layer": float(np.mean(cl)) if cl else None
    }

output_path = os.path.join(OUTPUT_DIR, "eas_full_results.json")
with open(output_path, "w") as f:
    json.dump(eas_output, f, indent=2)

print(f"EAS results saved to {output_path}\n")
print("FINAL SUMMARY")
print("=" * 50)
for label in ["yes", "no"]:
    s = eas_output["summary"]["by_label"][label]
    print(f"\n  {label} (n={s['n']}):")
    print(f"    Collapse rate:        {s['collapse_count']}/{s['n']} ({s['collapse_rate']:.1%})")
    print(f"    Mean EAS:             {s['mean_eas']:.4f}")
    print(f"    Mean final layer EAS: {s['mean_final_eas']:.4f}")
    if s["mean_collapse_layer"] is not None:
        print(f"    Mean collapse layer:  {s['mean_collapse_layer']:.1f}")
print("\n08_eas_assembly: COMPLETE")

In [ ]:
LATE_START = 24

late_slopes_yes = []
late_slopes_no  = []

for i, row in enumerate(aligned):
    trajectory = eas_arr[i, LATE_START:]
    x = np.arange(len(trajectory), dtype=float)
    slope, _, _, _, _ = stats.linregress(x, trajectory)
    if row["label"] == "yes":
        late_slopes_yes.append(slope)
    elif row["label"] == "no":
        late_slopes_no.append(slope)

t_stat, p_t = ttest_ind(late_slopes_yes, late_slopes_no, equal_var=False)
u_stat, p_u = mannwhitneyu(late_slopes_yes, late_slopes_no, alternative="two-sided")

print(f"Late-layer EAS slope (layers {LATE_START}-{n_layers-1}):")
print(f"  yes: mean slope = {np.mean(late_slopes_yes):+.4f}  (n={len(late_slopes_yes)})")
print(f"  no:  mean slope = {np.mean(late_slopes_no):+.4f}   (n={len(late_slopes_no)})")
print(f"  Welch t-test: t={t_stat:.3f}, p={p_t:.4f}{'  *' if p_t < 0.05 else ''}")
print(f"  Mann-Whitney: U={u_stat:.0f},  p={p_u:.4f}{'  *' if p_u < 0.05 else ''}")

In [ ]:
eas_no_attn = 0.5 * z_prob + 0.5 * z_ratio

print("Final-layer EAS (no attention term, W2=W3=0.5):")
for label in ["yes", "no"]:
    mask = np.array([r["label"] == label for r in aligned])
    vals = eas_no_attn[mask, -1]
    print(f"  {label}: {vals.mean():.4f}  (n={mask.sum()})")

t, p_t = ttest_ind(eas_no_attn[yes_mask, -1], eas_no_attn[no_mask, -1], equal_var=False)
u, p_u = mannwhitneyu(eas_no_attn[yes_mask, -1], eas_no_attn[no_mask, -1], alternative="two-sided")
print(f"  Welch: t={t:.3f}, p={p_t:.4f}")
print(f"  MWU:   U={u:.0f},  p={p_u:.4f}")

print("\nT4/T3 ratio at final layer only:")
for label in ["yes", "no"]:
    mask = np.array([r["label"] == label for r in aligned])
    vals = ratio_arr[mask, -1]
    print(f"  {label}: {vals.mean():.4f}  (n={mask.sum()})")

t, p_t = ttest_ind(ratio_arr[yes_mask, -1], ratio_arr[no_mask, -1], equal_var=False)
u, p_u = mannwhitneyu(ratio_arr[yes_mask, -1], ratio_arr[no_mask, -1], alternative="two-sided")
print(f"  Welch: t={t:.3f}, p={p_t:.4f}")
print(f"  MWU:   U={u:.0f},  p={p_u:.4f}")

In [ ]:
print("T2 prob_mass at final layer only:")
for label in ["yes", "no"]:
    mask = np.array([r["label"] == label for r in aligned])
    vals = prob_arr[mask, -1]
    print(f"  {label}: {vals.mean():.6f}  (n={mask.sum()})")

t, p_t = ttest_ind(prob_arr[yes_mask, -1], prob_arr[no_mask, -1], equal_var=False)
u, p_u = mannwhitneyu(prob_arr[yes_mask, -1], prob_arr[no_mask, -1], alternative="two-sided")
print(f"  Welch: t={t:.3f}, p={p_t:.4f}")
print(f"  MWU:   U={u:.0f},  p={p_u:.4f}")

print("\nT4 KL at final layer only:")
for label in ["yes", "no"]:
    mask = np.array([r["label"] == label for r in aligned])
    vals = t4_arr[mask, -1]
    print(f"  {label}: {vals.mean():.6f}  (n={mask.sum()})")

t, p_t = ttest_ind(t4_arr[yes_mask, -1], t4_arr[no_mask, -1], equal_var=False)
u, p_u = mannwhitneyu(t4_arr[yes_mask, -1], t4_arr[no_mask, -1], alternative="two-sided")
print(f"  Welch: t={t:.3f}, p={p_t:.4f}")
print(f"  MWU:   U={u:.0f},  p={p_u:.4f}")

In [ ]:
from scipy.stats import permutation_test

yes_trajectories = eas_arr[yes_mask]
no_trajectories = eas_arr[no_mask]

from statsmodels.stats.multitest import multipletests

layer_pvals = []
for l in range(n_layers):
    _, p = mannwhitneyu(yes_trajectories[:, l],
                        no_trajectories[:, l],
                        alternative="two-sided")
    layer_pvals.append(p)

reject, pvals_corrected, _, _ = multipletests(layer_pvals, method='fdr_bh')
sig_layers_fdr = [l for l, r in enumerate(reject) if r]

print("Significant layers after FDR correction:", sig_layers_fdr)
for l in sig_layers_fdr:
    yes_mean = yes_trajectories[:, l].mean()
    no_mean = no_trajectories[:, l].mean()
    direction = "yes>no" if yes_mean > no_mean else "no>yes"
    print(f"  Layer {l:2d}: yes={yes_mean:.4f} no={no_mean:.4f} [{direction}] p_adj={pvals_corrected[l]:.4f}")

In [ ]:
from scipy.stats import pearsonr, spearmanr

mean_t3 = t3_arr.mean(axis=1)
mean_t4 = t4_arr.mean(axis=1)

r_pearson, p_pearson = pearsonr(mean_t3, mean_t4)
r_spearman, p_spearman = spearmanr(mean_t3, mean_t4)

print(f"T3 vs T4 correlation (all samples):")
print(f"  Pearson:  r={r_pearson:.4f},  p={p_pearson:.4f}")
print(f"  Spearman: r={r_spearman:.4f}, p={p_spearman:.4f}")

for label in ["yes", "no"]:
    mask = np.array([r["label"] == label for r in aligned])
    r_p, p_p = pearsonr(mean_t3[mask], mean_t4[mask])
    r_s, p_s = spearmanr(mean_t3[mask], mean_t4[mask])
    print(f"\n  {label}: Pearson r={r_p:.4f} p={p_p:.4f} | "
          f"Spearman r={r_s:.4f} p={p_s:.4f}")

median_t3 = np.median(mean_t3)
high_conf = mean_t4[mean_t3 >= median_t3]
low_conf  = mean_t4[mean_t3 <  median_t3]

t, p_t = ttest_ind(high_conf, low_conf, equal_var=False)
u, p_u = mannwhitneyu(high_conf, low_conf, alternative="two-sided")
print(f"\nHigh vs low parametric confidence → retrieval influence:")
print(f"  High conf mean T4: {high_conf.mean():.4f}")
print(f"  Low  conf mean T4: {low_conf.mean():.4f}")
print(f"  Welch: t={t:.3f}, p={p_t:.4f}")
print(f"  MWU:   U={u:.0f},  p={p_u:.4f}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json
import numpy as np
from scipy.stats import spearmanr, pearsonr, mannwhitneyu

DRIVE_BASE = "/content/drive/MyDrive/medrag"

with open(f"{DRIVE_BASE}/trial4_outputs/trial4_full_results.json") as f:
    t4_data = json.load(f)
with open(f"{DRIVE_BASE}/trial3_outputs/trial3_full_results.json") as f:
    t3_data = json.load(f)
with open(f"{DRIVE_BASE}/trial2_outputs/trial2_full_results.json") as f:
    t2_data = json.load(f)

t4_samples = t4_data["samples"]
t3_by_pubid = {r["pubid"]: r for r in t3_data["samples"]}
t4_by_pubid = {r["pubid"]: r for r in t4_samples}
t2_by_pubid = {r["pubid"]: r for r in t2_data["samples"]}

print(f"T4: {len(t4_samples)} samples loaded")
print(f"T3: {len(t3_data['samples'])} samples loaded")
print(f"T2: {len(t2_data['samples'])} samples loaded")

peak_layers = [r["max_kl_layer"] for r in t4_samples]
print(f"\nT4 peak layer mean:  {np.mean(peak_layers):.1f}")
print(f"T4 peak layer std:   {np.std(peak_layers):.1f}")
print(f"T4 peak layer range: {min(peak_layers)} - {max(peak_layers)}")

common = [r["pubid"] for r in t4_samples if r["pubid"] in t3_by_pubid]
mean_t3 = np.array([t3_by_pubid[p]["mean_kl"] for p in common])
mean_t4 = np.array([t4_by_pubid[p]["mean_kl"] for p in common])
labels  = np.array([t4_by_pubid[p]["label"]   for p in common])

r_p, p_p = pearsonr(mean_t3, mean_t4)
r_s, p_s = spearmanr(mean_t3, mean_t4)
print(f"\nRetrieval resistance (T3 vs T4 correlation):")
print(f"  Pearson:  r={r_p:.4f}, p={p_p:.4f}")
print(f"  Spearman: r={r_s:.4f}, p={p_s:.4f}")
for label in ["yes", "no"]:
    idx = [i for i, p in enumerate(common) if t4_by_pubid[p]["label"] == label]
    r, p = spearmanr(mean_t3[idx], mean_t4[idx])
    print(f"  {label}: r={r:.4f}, p={p:.4f}")

t3_finals = np.array([t3_by_pubid[p]["final_layer_kl"] for p in common])
t4_finals = np.array([t4_by_pubid[p]["final_layer_kl"] for p in common])
ratio = t4_finals / np.maximum(t3_finals, 1e-6)
yes_ratio = ratio[labels == "yes"]
no_ratio  = ratio[labels == "no"]
u, p = mannwhitneyu(yes_ratio, no_ratio, alternative="two-sided")
print(f"\nT4/T3 ratio at final layer:")
print(f"  yes: {yes_ratio.mean():.4f}  no: {no_ratio.mean():.4f}")
print(f"  MWU p={p:.4f}")

common_all = [r["pubid"] for r in t4_samples
              if r["pubid"] in t3_by_pubid and r["pubid"] in t2_by_pubid]
t2_finals_arr = np.array([t2_by_pubid[p]["final_layer_prob_mass"] for p in common_all])
t4_finals_arr = np.array([t4_by_pubid[p]["final_layer_kl"]        for p in common_all])
labels_all    = np.array([t4_by_pubid[p]["label"]                 for p in common_all])

t2_norm = (t2_finals_arr - t2_finals_arr.min()) / (t2_finals_arr.max() - t2_finals_arr.min())
t4_norm = (t4_finals_arr - t4_finals_arr.min()) / (t4_finals_arr.max() - t4_finals_arr.min())
composite = 0.5 * t2_norm + 0.5 * t4_norm

tertiles = np.percentile(composite, [33, 66])
print(f"\nComposite EAS tertile analysis:")
for name, mask in [
    ("Low EAS",  composite <= tertiles[0]),
    ("Mid EAS",  (composite > tertiles[0]) & (composite <= tertiles[1])),
    ("High EAS", composite > tertiles[1])
]:
    yes_rate = (labels_all[mask] == "yes").mean()
    print(f"  {name} (n={mask.sum()}): yes_rate={yes_rate:.1%}")

In [ ]:
ev_ev_arr = np.array([r["t1"]["ev_to_ev_trajectory"] for r in aligned])

print("Evidence-to-evidence attention by label:")
for label in ["yes", "no"]:
    mask = np.array([r["label"] == label for r in aligned])
    vals = ev_ev_arr[mask].mean(axis=1)
    print(f"  {label}: mean={vals.mean():.6f} std={vals.std():.6f} n={mask.sum()}")

t, p_t = ttest_ind(ev_ev_arr[yes_mask].mean(axis=1),
                    ev_ev_arr[no_mask].mean(axis=1), equal_var=False)
u, p_u = mannwhitneyu(ev_ev_arr[yes_mask].mean(axis=1),
                       ev_ev_arr[no_mask].mean(axis=1), alternative="two-sided")
print(f"  Welch: t={t:.3f}, p={p_t:.4f}")
print(f"  MWU:   U={u:.0f}, p={p_u:.4f}")

peak_ev_ev = ev_ev_arr.argmax(axis=1)
for label in ["yes", "no"]:
    mask = np.array([r["label"] == label for r in aligned])
    print(f"  {label} peak layer: {peak_ev_ev[mask].mean():.1f}")

In [ ]:
supplementary = {
    "retrieval_resistance": {
        "spearman_r": -0.0980,
        "spearman_p": 0.0069,
        "pearson_r": 0.0004,
        "pearson_p": 0.9918,
        "by_label": {
            "yes": {"spearman_r": -0.0864, "p": 0.0616},
            "no":  {"spearman_r": -0.1130, "p": 0.0590}
        },
        "interpretation": "Weak negative correlation between parametric confidence and retrieval influence"
    },
    "tertile_analysis": {
        "low_eas_yes_rate":  0.586,
        "mid_eas_yes_rate":  0.584,
        "high_eas_yes_rate": 0.682,
        "interpretation": "High EAS correlates with yes-label queries, reflects T2/T4 directional differences"
    },
    "fdr_corrected_layers": {
        "significant_layers": [],
        "interpretation": "No individual layers survive FDR correction, evidence integration is distributed"
    },
    "late_layer_slope": {
        "yes_mean": 0.0645,
        "no_mean": 0.0641,
        "p_welch": 0.9161,
        "p_mwu": 0.3611,
        "interpretation": "Slopes nearly identical, no differential late-layer attenuation"
    },
    "ev_to_ev_attention": {
        "yes_mean": 0.001264,
        "no_mean": 0.001235,
        "p_welch": 0.3183,
        "p_mwu": 0.4763,
        "yes_peak_layer": 31.0,
        "no_peak_layer": 31.0,
        "interpretation": "No discriminative signal in evidence-to-evidence coherence"
    }
}

supp_path = os.path.join(OUTPUT_DIR, "eas_supplementary_diagnostics.json")
with open(supp_path, "w") as f:
    json.dump(supplementary, f, indent=2)
print(f"Saved to {supp_path}")